# DOHMH New York City Restaurant Inspection Results

In this file, we analyze the behavior and performance of the self-balancing sampler on the NYC [Restaurant Inspection Dataset](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j/about_data) which is available through [NYC Open Data](https://opendata.cityofnewyork.us/).

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

import os
from pathlib import Path

workdir = Path(os.getcwd())
data_dir = workdir / 'data'

## Data Schema

The *data/RestaurantInspectionDataDictionary_09242018.xlsx* file gives terse descriptions of the columns in this dataset while the *data/About_NYC_Restaurant_Inspection_Data_on_NYC_OpenData_050222.docx* file gives a broader exposition about what the data set contains. These encapsulate features like the location of the restauarant, contact information, the inspection date, inspection scores, and whatnot. There are a few features we are crucially interested in:

* **CAMIS**: A unique identifier for a specific restaurant.
* **Inspection Date**: Date inspection occurred.
* **Score** and **Grade**: A numerical score in $[0, \infty)$ and its corresponding letter grade. Points are awarded for infractions, so low scores are better. Letter grades are assigned from these scores typically as follows $[0, 13] \mapsto A$, $[14, 27] \mapsto B$, and $[28, \infty) \mapsto C$. 

One crucial omission from the dataset: restaurants that go out of business are **removed from the dataset**. We must be a bit careful to handle this sort of surviorship bias.

In [9]:
data_path = data_dir / 'DOHMH_New_York_City_Restaurant_Inspection_Results_20260512.csv'
full_df = pd.read_csv(data_path)

In [ ]:
display(list(full_df.columns))

['CAMIS',
 'DBA',
 'BORO',
 'BUILDING',
 'STREET',
 'ZIPCODE',
 'PHONE',
 'CUISINE DESCRIPTION',
 'INSPECTION DATE',
 'ACTION',
 'VIOLATION CODE',
 'VIOLATION DESCRIPTION',
 'CRITICAL FLAG',
 'SCORE',
 'GRADE',
 'GRADE DATE',
 'RECORD DATE',
 'INSPECTION TYPE',
 'Latitude',
 'Longitude',
 'Community Board',
 'Council District',
 'Census Tract',
 'BIN',
 'BBL',
 'NTA',
 'Location']

Filter to relevant columns of the dataset

In [21]:
# filter dataset to include only relevent columns
columns_to_keep = ['CAMIS', 'INSPECTION DATE', 'GRADE', 'SCORE', 'INSPECTION TYPE']
rename_map = {'CAMIS':'id', 'INSPECTION DATE':'date', 'GRADE':'grade', 'SCORE':'score', 'INSPECTION TYPE':'type'}
df = full_df[columns_to_keep].copy()
df.rename(columns=rename_map, inplace=True)

display(df.head())
print(df.shape)
print(df.dtypes)

,id,date,grade,score,type
0,50180724,01/01/1900,NaN,NaN,NaN
1,50166635,01/01/1900,NaN,NaN,NaN
2,50176620,01/01/1900,NaN,NaN,NaN
3,50179745,01/01/1900,NaN,NaN,NaN
4,50164534,01/01/1900,NaN,NaN,NaN


(296210, 5)
id         int64
date      object
grade     object
score    float64
type      object
dtype: object


Drop rows with missing numerical score AND missing letter grade. Fill missing grades according to the given map from scores to grades. 

In [ ]:
# drop missing rows
mask = df['grade'].isna() & df['score'].isna()
df_filtered = df[~mask]

# fill missing grades using scores
def score_mapper(score):
    '''Converts numerical score to letter grade.'''
    if score <= 13:
        return 'A'
    elif score <= 27:
        return 'B'
    else:
        return 'C'
    
mask = df['score'].isna()
df_filtered.loc[~mask, 'grade'] = df_filtered.loc[~mask, 'score'].apply(score_mapper)
df_filtered.head()

,id,date,grade,score,type
15,41304673,03/21/2026,C,34.0,Cycle Inspection / Initial Inspection
19,50136100,06/12/2025,B,24.0,Cycle Inspection / Initial Inspection
21,50032642,12/17/2024,A,12.0,Cycle Inspection / Re-inspection
29,50141096,10/13/2024,A,0.0,Pre-permit (Operational) / Initial Inspection
34,50086390,09/09/2024,A,0.0,Inter-Agency Task Force / Initial Inspection
